# V13: 모델 앙상블(XGB+LGBM) 및 교차 검증을 통한 안정성 강화

V12의 성능 저하를 해결하기 위해, 검증된 XGBoost와 LightGBM을 결합하고 5-Fold 교차 검증을 도입합니다.

## 주요 전략
1. **모델 앙상블**: XGBoost와 LightGBM 결합
2. **교차 검증**: 5-Fold CV로 일반화 성능 확보
3. **정밀도 유지**: 실수형(float) 예측값 유지

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold
from korean_font import set_korean_font
import warnings

set_korean_font()
warnings.filterwarnings('ignore')
plt.rcParams['axes.unicode_minus'] = False

한글 폰트 설정: Malgun Gothic (c:/Windows/Fonts/malgun.ttf)


In [2]:
# 1. 데이터 로드
train = pd.read_csv('data/train_median.csv')
test = pd.read_csv('data/test.csv')
weather_train = pd.read_csv('data/weather.csv')
weather_test = pd.read_csv('data/weatherTest.csv')

# 2. 날씨 통합
weather = pd.concat([weather_train, weather_test], axis=0)
weather['일시'] = pd.to_datetime(weather['일시'])
weather['비온날'] = (weather['강수량'].fillna(0) > 0).astype(int)
weather = weather[['일시', '기온', '습도', '비온날']]

# 3. 전처리
def preprocessing(df):
    df['일자'] = pd.to_datetime(df['일자'])
    df = pd.merge(df, weather, left_on='일자', right_on='일시', how='left')
    df['month'] = df['일자'].dt.month
    df['weekday'] = df['일자'].dt.weekday
    df['day'] = df['일자'].dt.day
    df['available'] = df['본사정원수'] - (df['본사휴가자수'] + df['본사출장자수'] + df['현본사소속재택근무자수'])
    
    df = df.sort_values('일자')
    df['holiday_before'] = (df['일자'].shift(-1).diff().dt.days > 2).astype(int)
    df['holiday_after'] = (df['일자'].diff().dt.days > 2).astype(int)
    
    meat_list = ['돈까스', '제육', '불고기', '고기', '치킨']
    df['meat_menu'] = df['중식메뉴'].apply(lambda x: sum(1 for m in meat_list if m in str(x)))
    return df.sort_index().fillna(0)

train_df = preprocessing(train)
test_df = preprocessing(test)
train_df['target_l'] = train_df['중식계'] / train_df['available']
train_df['target_d'] = train_df['석식계'] / train_df['available']

In [3]:
# 4. 모델링
features = ['month', 'weekday', 'day', 'available', '본사출장자수', '본사시간외근무명령서승인건수', 
            '기온', '습도', '비온날', 'holiday_before', 'holiday_after', 'meat_menu']

def run_kfold_ensemble(y_series):
    X = train_df[features]
    y = y_series
    X_test = test_df[features]
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    final_preds = np.zeros(len(X_test))
    
    for tr_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        
        m_xgb = XGBRegressor(n_estimators=1000, learning_rate=0.03, max_depth=6, random_state=42)
        m_xgb.fit(X_tr, y_tr)
        
        m_lgbm = LGBMRegressor(n_estimators=1000, learning_rate=0.03, max_depth=6, random_state=42, verbose=-1)
        m_lgbm.fit(X_tr, y_tr)
        
        final_preds += (m_xgb.predict(X_test) + m_lgbm.predict(X_test)) / 10
    return final_preds

res_lunch = run_kfold_ensemble(train_df['target_l']) * test_df['available']
res_dinner = run_kfold_ensemble(train_df['target_d']) * test_df['available']

In [4]:
# 5. 저장
sample = pd.read_csv('data/sample_submission.csv')
sample['중식계'] = sample['중식계'].astype(float)
sample['석식계'] = sample['석식계'].astype(float)

test_df['date_str'] = test_df['일자'].dt.strftime('%Y-%m-%d')
result_map = pd.DataFrame({'date': test_df['date_str'], 'L': res_lunch, 'D': res_dinner})

for i, row in sample.iterrows():
    match = result_map[result_map['date'] == row['일자']]
    if not match.empty:
        sample.iloc[i, 1] = match['L'].values[0]
        sample.iloc[i, 2] = match['D'].values[0]

sample.to_csv('submission/submission_v13.csv', index=False, encoding='utf-8-sig')
print('v13 저장 완료')

v13 저장 완료
